# ExpenseLM — Colab GPU notebook (E0, E1, SFT→E2, DPO→E3, GGUF export)

**Runtime → Change runtime type → T4 GPU** before running anything.

This notebook is deliberately thin: every cell delegates to the scripts in the repo,
so there is exactly ONE source of truth for prompts, grading, and hyperparameters.

**Before this notebook** (on your Mac): dataset built, verified, decontaminated —
and the repo folder (with `data/splits/`) uploaded to Google Drive as `MyDrive/expenselm`.
See `docs/RUNBOOK.md` Phase 1–2.

In [1]:
# 0.1 — confirm the GPU
!nvidia-smi | head -12

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# 0.2 — pull the repo from Drive into the fast local disk
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/expenselm /content/expenselm
%cd /content/expenselm

In [ ]:
# 0.3 — install. unsloth first (it patches transformers/trl), then the repo itself.
!pip install -q unsloth trl peft bitsandbytes wandb
!pip uninstall -y torchao   # Colab's torchao 0.10 breaks peft LoRA load; unused here
!pip install -q -e .
import wandb; wandb.login()  # paste your W&B key (wandb.ai/authorize) — loss curves for the report

## Week 3 — baselines E0 / E1 (no training yet — this is the floor)

Tip: first run with `--limit 10` as a smoke test (~2 min), then the full 200.
Full runs take ~30–60 min each on a T4 (unbatched greedy decoding — fine, run it once).

In [ ]:
!python -m expenselm.cli eval --system e0 --split data/splits/test.jsonl --limit 10   # smoke first
!python -m expenselm.cli eval --system e0 --split data/splits/test.jsonl             # then full

In [ ]:
!python -m expenselm.cli eval --system e1 --split data/splits/test.jsonl --dev-file data/splits/dev.jsonl

## Week 4 — SFT (QLoRA via Unsloth) → E2

Read `docs/LEARNING_GUIDE.md` §7–8 first, and the hyperparameter comments in
`src/expenselm/train/sft_unsloth.py` — Week 4's definition of done is being able
to explain every one of them.

**Colab-death insurance (PRD §9):** change `output_dir="outputs"` in the script to
`/content/drive/MyDrive/expenselm-runs/sft` so checkpoints survive a disconnect.

In [ ]:
# render training files with the model's chat template (same builder the eval uses)
!python -m expenselm.cli format --in data/splits/train.jsonl --out train.jsonl
!python -m expenselm.cli format --in data/splits/dev.jsonl --out dev_chatml.jsonl
%run src/expenselm/train/sft_unsloth.py

In [ ]:
# E2 row + back up the adapter to Drive
!mkdir -p models /content/drive/MyDrive/expenselm-runs
# find the adapter wherever training saved it (CWD-independent), then place it where eval expects
!cp -r "$(dirname $(find /content -path '*sft-adapter*' -name adapter_config.json 2>/dev/null | head -1))" models/sft-adapter
!cp -r models/sft-adapter /content/drive/MyDrive/expenselm-runs/
!python -m expenselm.cli eval --system e2 --split data/splits/test.jsonl

### One raw-TRL reference mini-run (so Unsloth isn't magic)
Fresh runtime recommended (Runtime → Restart) so peak-VRAM numbers are clean.
Uses ~200 examples; record tokens/sec + peak VRAM and compare with the Unsloth run.

In [ ]:
!head -200 train.jsonl > train_small.jsonl
%run src/expenselm/train/sft_trl_reference.py

## Week 5 — harvest real SFT failures → DPO → E3

In [ ]:
!python -m expenselm.train.harvest_failures \
    --splits data/splits/dev.jsonl data/splits/train.jsonl \
    --out dpo_pairs.jsonl --target 500 --adapter models/sft-adapter

In [ ]:
%run src/expenselm/train/dpo_unsloth.py

In [ ]:
!cp -r dpo-adapter models/ && cp -r dpo-adapter /content/drive/MyDrive/expenselm-runs/
!python -m expenselm.cli eval --system e3 --split data/splits/test.jsonl
# ALSO: eval e2 and e3 on the DEV set and diff per failure mode before believing E3.
!python -m expenselm.cli eval --system e3 --split data/splits/dev.jsonl --limit 100

## Week 6 — merge + GGUF export (then E5 happens on your Mac)
See `src/expenselm/train/export_gguf.md` for the concepts (merging, Q4_K_M vs NF4).

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained('dpo-adapter', max_seq_length=2048, load_in_4bit=True)
model.save_pretrained_gguf('expenselm-gguf', tokenizer, quantization_method='q4_k_m')
model.save_pretrained_gguf('expenselm-gguf-q8', tokenizer, quantization_method='q8_0')  # control point
!cp expenselm-gguf*/*.gguf /content/drive/MyDrive/expenselm-runs/

In [ ]:
# copy the metrics home — results/*.json feed `expenselm report` on your Mac
!cp -r results /content/drive/MyDrive/expenselm-runs/results